# Ticket 5: quality gate + dq_violations

Scope: `company_code = 1000, fiscal_year = 2024, fiscal_period = 1`, the
same rows `fact_gl_line` already holds from ticket 4. Checking how many
of the 4 blocking checks and 2 non-blocking findings have a real case in
this data before designing the gate.

In [1]:
import duckdb

con = duckdb.connect("../warehouse.duckdb", read_only=True)
SCOPE = "company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = 1"

In [2]:
# null in a key column (grain columns plus gl_account, the account used for mapping)
con.execute(f"""
    SELECT COUNT(*) FROM stg_gl WHERE {SCOPE} AND (
        company_code IS NULL OR document_id IS NULL OR line_number IS NULL OR
        fiscal_year IS NULL OR fiscal_period IS NULL OR gl_account IS NULL
    )
""").fetchall()

[(0,)]

**Zero.** No real case this period. Needs a synthetic-row proof, same
pattern as ticket 4's `unmapped_doc_type` check.

In [3]:
# duplicate grain: the same company + document + line + year + period more than once
con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT company_code, document_id, line_number, fiscal_year, fiscal_period, COUNT(*) n
        FROM stg_gl WHERE {SCOPE} GROUP BY 1,2,3,4,5 HAVING n > 1
    )
""").fetchall()

[(0,)]

**Zero.** Same as ticket 4's finding for this period. Needs a synthetic
proof too.

In [4]:
import csv

# map_fanout: source_account repeated in map_account.csv
with open("../map_account.csv", newline="") as f:
    rows = list(csv.DictReader(f))
accounts = [r["source_account"] for r in rows]
len(rows), len(set(accounts))

(505, 505)

**505 rows, 505 distinct.** No fanout right now, `src/checks.py` already
guards this at mapping-build time. The gate re-checks it at load time too
(docs/definitions.md: "checked before every load", not just once).

In [5]:
# unbalanced_document: same check ticket 4 already runs inline
con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT document_id FROM stg_gl WHERE {SCOPE}
        GROUP BY 1 HAVING ROUND(SUM(debit_amount) - SUM(credit_amount), 2) != 0
    )
""").fetchall()

[(1,)]

**1 document**, the same one ticket 4 found and excluded. This check
already has real behavior to move from `load_fact.py`'s inline SQL into
a shared gate.

In [6]:
# local_amount_imbalance: balances on debit/credit but local_amount doesn't net to zero
con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT document_id,
               ROUND(SUM(debit_amount) - SUM(credit_amount), 2) AS dc_gap,
               ROUND(SUM(local_amount), 2) AS local_net
        FROM stg_gl WHERE {SCOPE}
        GROUP BY 1 HAVING dc_gap = 0 AND local_net != 0
    )
""").fetchall()

[(23,)]

**23 documents.** A real, sizeable non-blocking finding: balanced on
debit/credit, but `local_amount` doesn't net to zero (currency-conversion
rounding, per `docs/definitions.md`). Worth its own row per document in
`dq_violations`, not just a count.

In [7]:
# unmapped_account: fact_gl_line rows whose map_status isn't mapped
con.execute("SELECT map_status, COUNT(*) FROM fact_gl_line GROUP BY 1 ORDER BY 2 DESC").fetchall()

[('mapped', 13055), ('catch_all', 70), ('unmapped', 15)]

`mapped=13055, catch_all=70, unmapped=15`. Per `docs/definitions.md`,
`unmapped_account` means `status != mapped`, so both `catch_all` and
`unmapped` rows count: **85 lines**, already loaded (ticket 4 never
filtered on `map_status`), just not flagged as a finding yet.

## What I've got

| check | blocking | real cases this period |
|---|---|---|
| `unbalanced_document` | yes | 1 document (already excluded by ticket 4) |
| `duplicate_source` | yes | 0, needs a synthetic proof |
| `map_fanout` | yes | 0, needs a synthetic proof |
| null key column | yes | 0, needs a synthetic proof |
| `local_amount_imbalance` | no | 23 documents |
| `unmapped_account` | no | 85 lines |

`unbalanced_document` already has real, working logic inline in
`load_fact.py`. The gate's job is to pull it out into one shared module
the other three blocking checks join, and to log every finding, blocking
and non-blocking, into one `dq_violations` table instead of the narrower
`fact_gl_line_rejected` ticket 4 built as a placeholder.